# Simulation Inspection

This notebook inspects the antibiotic resistance simulation before reinforcement-learning training.

The purpose is to verify that the current simulation components behave as intended and to expose unresolved modeling boundaries.

Transition probabilities are computational scenario assumptions, not empirical biological probabilities. `REF_UNIFORM_SUPPORTED` selects uniformly among applicable deduplicated candidates and selects one whenever that set is nonempty. Named sensitivity scenarios use illustrative transition-occurrence assumptions (`q = 0.25`, `0.50`, and `0.75`).

In [1]:
import os, sys
print(os.getcwd())
print(sys.executable)

c:\Devtools\source\personal\Capstone-101\experiments
c:\Devtools\source\personal\Capstone-101\.venv\Scripts\python.exe


In [ ]:
%cd ..
from simulation.action_space import ActionSpace

c:\Devtools\source\personal\Capstone-101


In [ ]:
from simulation.action_space import ActionSpace
from simulation.candidate_transition import CandidateTransition
from simulation.decoder import decode_observation
from simulation.encoder import encode_observation
from simulation.environment import AntibioticEnvironment
from simulation.episode import EpisodeState
from simulation.resistance_state import ResistanceState
from simulation.reward import RewardFunction
from simulation.stochastic_transition_model import StochasticTransitionModel
from simulation.transition_function import ResistanceTransitionFunction
from simulation.transition_generator import CandidateTransitionGenerator
from simulation.transition_sampler import TransitionSampler
from simulation.treatment_action_executor import TreatmentActionExecutor

## 1. Inspect the action space

The MVP contains seven antibiotics.

The action IDs must remain aligned with the canonical action-space dataset and resistance-state representation.

In [ ]:
action_space = ActionSpace()

for action_id, antibiotic in enumerate(action_space.actions):
    print(f"{action_id}: {antibiotic}")

print(f"\nNumber of actions: {action_space.size}")

0: CIPROFLOXACIN
1: NITROFURANTOIN
2: FOSFOMYCIN
3: TRIMETHOPRIM
4: GENTAMICIN
5: MECILLINAM
6: CEFTAZIDIME

Number of actions: 7


## 2. Inspect the initial bacterial state

The current MVP uses a binary seven-dimensional resistance state.

`0` = susceptible  
`1` = resistant

The initial reference state is fully susceptible.

In [ ]:
initial_state = ResistanceState(
    (0, 0, 0, 0, 0, 0, 0)
)

print("Resistance vector:", initial_state.resistance)
print("Resistant antibiotics:", initial_state.resistant_antibiotics())

Resistance vector: (0, 0, 0, 0, 0, 0, 0)
Resistant antibiotics: ()


## 3. Inspect treatment effectiveness

Treatment effectiveness is defined computationally as whether the selected antibiotic is susceptible in the current resistance state.

This is not equivalent to clinical cure.

In [ ]:
treatment_executor = TreatmentActionExecutor(action_space)

for action in range(action_space.size):
    result = treatment_executor.execute(initial_state, action)

    print(
        f"{result.antibiotic}: "
        f"effective={result.effective}"
    )

CIPROFLOXACIN: effective=True
NITROFURANTOIN: effective=True
FOSFOMYCIN: effective=True
TRIMETHOPRIM: effective=True
GENTAMICIN: effective=True
MECILLINAM: effective=True
CEFTAZIDIME: effective=True


## 4. Inspect evidence-backed candidate transitions

Candidate generation converts supported empirical relationships into possible state transitions.

It does not assign probabilities.

A candidate is only generated when the current resistance state makes that transition possible.

For example:

- CR can move a currently susceptible target to resistant.
- CS can move a currently resistant target to susceptible.
- neutral leaves the target resistance state unchanged.

In [ ]:
generator = CandidateTransitionGenerator(action_space)

for action in range(action_space.size):
    candidates = generator.generate(initial_state, action)

    print(f"\nAction {action}: {action_space.antibiotic_for(action)}")

    if not candidates:
        print("  No currently applicable candidates.")
        continue

    for candidate in candidates:
        print(
            f"  {candidate.outcome}: "
            f"{candidate.target_drug} "
            f"source={candidate.source_ids}"
        )


Action 0: CIPROFLOXACIN
  cross_resistance: CEFTAZIDIME source=('POD2018',)
  cross_resistance: TRIMETHOPRIM source=('POD2018',)
  cross_resistance: MECILLINAM source=('POD2018',)

Action 1: NITROFURANTOIN
  cross_resistance: CIPROFLOXACIN source=('POD2018',)

Action 2: FOSFOMYCIN
  No currently applicable candidates.

Action 3: TRIMETHOPRIM
  cross_resistance: NITROFURANTOIN source=('JAMES2024',)
  neutral: FOSFOMYCIN source=('JAMES2024',)

Action 4: GENTAMICIN
  No currently applicable candidates.

Action 5: MECILLINAM
  cross_resistance: TRIMETHOPRIM source=('POD2018',)

Action 6: CEFTAZIDIME
  No currently applicable candidates.


## 5. Inspect a cross-resistance transition

We use a controlled state so that the target antibiotic is susceptible and an evidence-backed CR relationship can be applied.

This is a software fixture for inspecting the transition mechanism, not a claim about the probability of this biological event.

In [ ]:
cross_resistance_state = ResistanceState(
    (0, 0, 0, 0, 0, 0, 0)
)

candidates = generator.generate(
    cross_resistance_state,
    action_space.action_for("CIPROFLOXACIN"),
)

cr_candidates = tuple(
    candidate
    for candidate in candidates
    if candidate.outcome == "cross_resistance"
)

print("Cross-resistance candidates:")
for candidate in cr_candidates:
    print(candidate)

Cross-resistance candidates:
CandidateTransition(target_drug='CEFTAZIDIME', outcome='cross_resistance', source_ids=('POD2018',))
CandidateTransition(target_drug='TRIMETHOPRIM', outcome='cross_resistance', source_ids=('POD2018',))
CandidateTransition(target_drug='MECILLINAM', outcome='cross_resistance', source_ids=('POD2018',))


In [ ]:
transition_function = ResistanceTransitionFunction()

if cr_candidates:
    candidate = cr_candidates[0]

    next_state = transition_function.apply(
        cross_resistance_state,
        candidate,
    )

    print("Before:", cross_resistance_state.resistance)
    print("After: ", next_state.resistance)
    print("Outcome:", candidate.outcome)
    print("Target:", candidate.target_drug)
    print("Evidence:", candidate.source_ids)

Before: (0, 0, 0, 0, 0, 0, 0)
After:  (0, 0, 0, 0, 0, 0, 1)
Outcome: cross_resistance
Target: CEFTAZIDIME
Evidence: ('POD2018',)


## 6. Inspect a collateral-sensitivity transition

Collateral sensitivity requires the target antibiotic to be resistant in the current state.

We therefore construct a controlled state in which the target of a documented CS relationship is resistant.

Again, this demonstrates the transition mechanism rather than estimating an event probability.

In [ ]:
collateral_state = ResistanceState(
    (0, 0, 0, 0, 1, 0, 0)
)

candidates = generator.generate(
    collateral_state,
    action_space.action_for("CIPROFLOXACIN"),
)

cs_candidates = tuple(
    candidate
    for candidate in candidates
    if candidate.outcome == "collateral_sensitivity"
)

print("Collateral-sensitivity candidates:")
for candidate in cs_candidates:
    print(candidate)

Collateral-sensitivity candidates:
CandidateTransition(target_drug='GENTAMICIN', outcome='collateral_sensitivity', source_ids=('POD2018',))


In [ ]:
if cs_candidates:
    candidate = cs_candidates[0]

    next_state = transition_function.apply(
        collateral_state,
        candidate,
    )

    print("Before:", collateral_state.resistance)
    print("After: ", next_state.resistance)
    print("Outcome:", candidate.outcome)
    print("Target:", candidate.target_drug)
    print("Evidence:", candidate.source_ids)

Before: (0, 0, 0, 0, 1, 0, 0)
After:  (0, 0, 0, 0, 0, 0, 0)
Outcome: collateral_sensitivity
Target: GENTAMICIN
Evidence: ('POD2018',)


## 7. Inspect a neutral transition

A neutral relationship does not change the resistance state.

The simulator preserves this as an explicit transition outcome rather than treating every action as resistance-changing.

In [ ]:
neutral_state = ResistanceState(
    (0, 0, 0, 0, 0, 0, 0)
)

neutral_candidates = generator.generate(
    neutral_state,
    action_space.action_for("TRIMETHOPRIM"),
)

for candidate in neutral_candidates:
    if candidate.outcome == "neutral":
        next_state = transition_function.apply(
            neutral_state,
            candidate,
        )

        print("Before:", neutral_state.resistance)
        print("After: ", next_state.resistance)
        print("Outcome:", candidate.outcome)
        print("Target:", candidate.target_drug)
        print("Evidence:", candidate.source_ids)

Before: (0, 0, 0, 0, 0, 0, 0)
After:  (0, 0, 0, 0, 0, 0, 0)
Outcome: neutral
Target: FOSFOMYCIN
Evidence: ('JAMES2024',)


## 8. Inspect transition scenarios

`REF_UNIFORM_SUPPORTED` assigns equal scenario probability to each deduplicated applicable candidate and selects one whenever candidates exist. This deliberately forces a supported candidate outcome on each such step; the candidate-selection probability is not the biological probability that resistance changes.

Sensitivity scenarios separate occurrence from outcome selection. For resistance-changing candidates, `q` is the explicit computational occurrence assumption and candidates are selected uniformly conditional on occurrence. The configured values `0.25`, `0.50`, and `0.75` are illustrative assumptions, not evidence-derived estimates.

Neutral candidates remain supported unchanged outcomes. In sensitivity scenarios, they do not compete with resistance-changing candidates when any are applicable; a neutral-only set is sampled uniformly. An empty candidate set is reported as `unsupported_no_candidate`, distinct from a sampled neutral outcome or a sensitivity-gate `no_transition`.

In [ ]:
sampler = TransitionSampler()

single_candidate = CandidateTransition(
    target_drug="GENTAMICIN",
    outcome="cross_resistance",
    source_ids=("INSPECTION_FIXTURE",),
)

sampled = sampler.sample(
    (single_candidate,),
    seed=42,
)

print("Sampled candidate:")
print(sampled.candidate)

Sampled candidate:
CandidateTransition(target_drug='GENTAMICIN', outcome='cross_resistance', source_ids=('INSPECTION_FIXTURE',))


In [ ]:
multiple_candidates = (
    CandidateTransition(
        target_drug="GENTAMICIN",
        outcome="cross_resistance",
        source_ids=("SOURCE_A",),
    ),
    CandidateTransition(
        target_drug="FOSFOMYCIN",
        outcome="cross_resistance",
        source_ids=("SOURCE_B",),
    ),
)

sampled_multiple = sampler.sample(
    multiple_candidates,
    seed=42,
)

assert sampled_multiple.candidate in multiple_candidates
assert sampled_multiple.candidate_probabilities == (0.5, 0.5)
assert sum(sampled_multiple.candidate_probabilities) == 1.0
print("Reference candidates:", sampled_multiple.candidates)
print("Conditional candidate probabilities:", sampled_multiple.candidate_probabilities)
print("Selected candidate:", sampled_multiple.candidate)
print("Scenario:", sampled_multiple.scenario_id)
print("Status:", sampled_multiple.status)

In [ ]:
q_zero_sampler = TransitionSampler("SENSITIVITY_Q_000")
q_zero_result = q_zero_sampler.sample(
    multiple_candidates,
    seed=42,
)

assert q_zero_result.candidate is None
assert q_zero_result.status == "no_transition"
assert q_zero_result.no_transition_probability == 1.0
assert sum(q_zero_result.candidate_probabilities) + q_zero_result.no_transition_probability == 1.0
print("Sensitivity q=0 status:", q_zero_result.status)
print("Candidate probabilities:", q_zero_result.candidate_probabilities)
print("No-transition probability assumption:", q_zero_result.no_transition_probability)

q_quarter_result = TransitionSampler("SENSITIVITY_Q_025").sample(
    multiple_candidates,
    seed=42,
)
assert q_quarter_result.occurrence_probability == 0.25
assert q_quarter_result.candidate_probabilities == (0.125, 0.125)
assert q_quarter_result.no_transition_probability == 0.75
assert sum(q_quarter_result.candidate_probabilities) + q_quarter_result.no_transition_probability == 1.0
print("Sensitivity q=0.25 candidate probabilities:", q_quarter_result.candidate_probabilities)
print("Sensitivity q=0.25 no-transition assumption:", q_quarter_result.no_transition_probability)

unsupported_state, unsupported_result = StochasticTransitionModel(
    candidate_generator=generator
).step(initial_state, action=2, seed=42)

assert unsupported_state == initial_state
assert unsupported_result.candidates == ()
assert unsupported_result.candidate is None
assert unsupported_result.status == "unsupported_no_candidate"
assert unsupported_result.no_transition_probability is None
print("Empty candidate status:", unsupported_result.status)

neutral_candidate = CandidateTransition(
    target_drug="FOSFOMYCIN",
    outcome="neutral",
    source_ids=("JAMES2024",),
)
neutral_sampled = TransitionSampler("SENSITIVITY_Q_025").sample(
    (neutral_candidate,),
    seed=42,
)
assert neutral_sampled.status == "neutral"
assert neutral_sampled.candidate == neutral_candidate
print("Neutral-only sensitivity status:", neutral_sampled.status)

## 9. Inspect observation encoding and decoding

The current observation contains:

- susceptibility profile: 7 values
- previous action: 7 one-hot values
- treatment step: 1 value

Total observation size = 15.

The current MVP exposes the true susceptibility profile directly.

`-1` remains available for a future unknown/noisy observation model.

In [ ]:
environment = AntibioticEnvironment()

observation, info = environment.reset(seed=42)

print("Observation:")
print(observation)

print("\nShape:", observation.shape)
print("Dtype:", observation.dtype)

Observation:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Shape: (15,)
Dtype: float32


In [ ]:
decoded = decode_observation(
    tuple(observation.astype(int))
)

print("Decoded observation:")
print(decoded)

re_encoded = encode_observation(decoded)

print("\nRound-trip successful:", tuple(re_encoded) == tuple(
    observation.astype(int)
))

Decoded observation:
Observation(susceptibility=(np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), last_action=(np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)), treatment_step=np.int64(0))

Round-trip successful: True


## 10. Inspect the Gymnasium environment

The environment returns the `EpisodeStepResult` observation and reward and exposes transition provenance in `info`, including the scenario, transition status, selected outcome, evidence source IDs, candidate probabilities, and transition seed.

The candidate-selection probabilities are scenario assumptions, not biological probabilities.

In [ ]:
environment = AntibioticEnvironment()

observation, info = environment.reset(seed=42)
assert environment.observation_space.contains(observation)
print("Initial observation:", observation)

action = action_space.action_for("CIPROFLOXACIN")
next_observation, reward, terminated, truncated, info = environment.step(action)

assert environment.observation_space.contains(next_observation)
assert info["antibiotic"] == "CIPROFLOXACIN"
assert info["scenario_id"] == "REF_UNIFORM_SUPPORTED"
assert info["transition_status"] in {"cross_resistance", "collateral_sensitivity", "neutral"}
assert info["source_ids"]
assert isinstance(info["candidate_scenario_probabilities"], list)
print("Next observation:", next_observation)
print("Reward:", reward)
print("Terminated / truncated:", terminated, truncated)
print("Transition audit info:", info)

Initial observation: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Step result: (array([0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1.],
      dtype=float32), 0.4, False, False, {'antibiotic': 'CIPROFLOXACIN', 'effective': True})


## 11. Inspect the reward function

The reward currently combines:

- treatment effectiveness
- resistance change
- treatment exposure

The weights are configurable.

This reward is a computational objective for the simulator and is not a clinical utility function.

In [ ]:
reward_function = RewardFunction(
    action_space=action_space
)

previous_state = ResistanceState(
    (0, 0, 0, 0, 0, 0, 0)
)

next_state = ResistanceState(
    (0, 0, 0, 0, 1, 0, 0)
)

action = action_space.action_for("CIPROFLOXACIN")

reward = reward_function.calculate(
    previous_state,
    next_state,
    action,
)

print("Previous state:", previous_state.resistance)
print("Next state:    ", next_state.resistance)
print("Action:        ", action_space.antibiotic_for(action))
print("Reward:        ", reward)

Previous state: (0, 0, 0, 0, 0, 0, 0)
Next state:     (0, 0, 0, 0, 1, 0, 0)
Action:         CIPROFLOXACIN
Reward:         0.4


## 12. Inspect episode termination

The current MVP terminates when:

1. the maximum treatment horizon is reached, or
2. all seven antibiotics are resistant.

Clinical clearance is not currently modeled.

In [ ]:
from simulation.episode_termination import EpisodeTermination

termination = EpisodeTermination(
    action_space=action_space,
    max_steps=8,
)

initial_episode = EpisodeState(
    resistance_state=ResistanceState(
        (0, 0, 0, 0, 0, 0, 0)
    ),
    treatment_step=0,
)

print(
    "Initial episode terminated:",
    termination.is_terminated(initial_episode)
)

Initial episode terminated: False


In [ ]:
max_step_episode = EpisodeState(
    resistance_state=ResistanceState(
        (0, 0, 0, 0, 0, 0, 0)
    ),
    treatment_step=8,
)

all_resistant_episode = EpisodeState(
    resistance_state=ResistanceState(
        (1, 1, 1, 1, 1, 1, 1)
    ),
    treatment_step=3,
)

print(
    "Maximum-step termination:",
    termination.is_terminated(max_step_episode)
)

print(
    "All-resistant termination:",
    termination.is_terminated(all_resistant_episode)
)

Maximum-step termination: True
All-resistant termination: True


## 13. Inspection summary

### Confirmed

- Seven-antibiotic action space is loaded in canonical order.
- Resistance state is a validated seven-dimensional binary state.
- Treatment effectiveness is derived from susceptibility and is not clinical cure.
- Evidence-backed cross-resistance, collateral-sensitivity, and neutral outcomes can be inspected.
- The primary `REF_UNIFORM_SUPPORTED` scenario uniformly selects one applicable candidate whenever candidates exist; this is a transition-forcing computational reference, not a biological occurrence probability.
- Named sensitivity scenarios use illustrative `q` assumptions and keep neutral, no-transition, and unsupported-no-candidate statuses distinct.
- The environment preserves the episode-step observation, reward, and transition provenance in its outputs.
- Observation encoding produces a 15-value vector and round-trips successfully.
- Episode termination and reward behavior can be inspected independently.

### Interpretation

The `q` sensitivity values are assumptions for robustness analysis, not empirical estimates. Neither the reference scenario nor its sensitivity settings provide clinical validation or calibrated predictions of bacterial evolution.